# Demo 07c — Bagging and Random Forests

**Class 07 · E3** (slides: `slides/07_ensembles.md`)

A vote needs different members. **Bagging** makes them from one learner: each tree sees its own bootstrap sample. A **random forest** also hides features at each split. The running example is the checkerboard, where the shallow tree is weak and bagging wins.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

from faa import boundaries, data, demo, metrics, plotlib
from faa import galleries as ga
from faa import synthetic as sy
from faa.demo import show, slide
from faa.galleries import CLASS_SETS, MEMBERS, REG_SETS, board, family_figure, names_of
from faa.metrics import mcc
from faa.plotlib import BLUE, GREY, ORANGE, RED

demo.setup("07")
SEED = 0
N_JOBS = 8

X_tr, y_tr, X_te, y_te = sy.load("checkerboard", SEED)
LIM = boundaries.plane_limits(X_tr, pad=0.25)

In [ ]:
slide("The Bootstrap Sample: 63.2 % of the Points")
rng = np.random.default_rng(SEED)
rows = []
for n in (10, 50, 210, 1000, 10_000):
    frac = np.mean([len(np.unique(rng.integers(0, n, n))) / n for _ in range(200)])
    rows.append((n, float(frac), 1 - (1 - 1 / n) ** n))
show(pl.DataFrame(rows, schema=["n", "simulated (200 draws)", "1 - (1 - 1/n)^n"], orient="row"), 4)
print(
    f"limit: 1 - 1/e = {1 - math.exp(-1):.4f}; the share of a training set that one tree never sees (out of bag): 1/e = {math.exp(-1):.4f}"
)
idx = rng.integers(0, 10, 10)
print(
    f"one bootstrap sample of n = 10 (indices): {sorted(idx.tolist())}: {len(np.unique(idx))} distinct, {10 - len(np.unique(idx))} left out"
)

In [ ]:
# plot e3_share
fg, ax = plotlib.fig("full", 1.1)
ns = np.array([2, 3, 5, 10, 20, 50, 100, 1000, 10_000])
ax.plot(ns, 1 - (1 - 1 / ns) ** ns, color=BLUE, marker="o", ms=2.5)
ax.axhline(1 - math.exp(-1), color=GREY, lw=0.7, ls=":")
ax.text(2, 0.637, "limit 0.632", fontsize=6.3, color=GREY, va="bottom")
ax.set(xscale="log", xlabel="examples $n$ (log scale)", ylabel="distinct share", ylim=(0.6, 0.78))
plotlib.save(fg, "e3_share")

In [ ]:
slide("Bagging by Hand")


class HandBag:
    """Bagging of unpruned trees in 12 lines: T bootstrap samples, one tree each, the mean of the T probabilities."""

    def __init__(self, n_trees: int, seed: int = 0, max_features=None):
        self.n_trees, self.seed, self.max_features = n_trees, seed, max_features

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n = len(y)
        self.trees_, self.in_bag_ = [], []
        for t in range(self.n_trees):
            idx = rng.integers(0, n, n)  # a bootstrap sample: n draws with replacement
            tree = DecisionTreeClassifier(max_features=self.max_features, random_state=self.seed * 1000 + t)
            self.trees_.append(tree.fit(X[idx], y[idx]))
            self.in_bag_.append(np.isin(np.arange(n), idx))
        return self

    def predict_proba(self, X):
        p1 = np.mean([t.predict_proba(X)[:, 1] for t in self.trees_], axis=0)
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def oob_mcc(self, X, y, upto: int | None = None) -> float:
        """MCC of the vote of the trees that did not see each point (points that every tree saw are skipped)."""
        T = self.n_trees if upto is None else upto
        votes, counts = np.zeros(len(y)), np.zeros(len(y))
        for t in range(T):
            out = ~self.in_bag_[t]
            votes[out] += self.trees_[t].predict_proba(X[out])[:, 1]
            counts[out] += 1
        seen = counts > 0
        return mcc(y[seen], votes[seen] / counts[seen] > 0.5)

In [ ]:
hand = HandBag(100, SEED).fit(X_tr, y_tr)
sk_bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=SEED).fit(X_tr, y_tr)
sk_rf = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_tr, y_tr)
single = DecisionTreeClassifier(random_state=SEED).fit(X_tr, y_tr)
rows = [
    ("one deep tree", mcc(y_te, single.predict(X_te)), float("nan")),
    ("bagging by hand, 100 trees", mcc(y_te, hand.predict(X_te)), hand.oob_mcc(X_tr, y_tr)),
    ("BaggingClassifier, 100 trees", mcc(y_te, sk_bag.predict(X_te)), float("nan")),
    ("RandomForestClassifier, 100 trees", mcc(y_te, sk_rf.predict(X_te)), float("nan")),
]
show(pl.DataFrame(rows, schema=["model", "test MCC", "OOB MCC"], orient="row"))

In [ ]:
slide("Bagged Trees: 1, 3, 10 and 100 Trees")
bags = {T: HandBag(T, SEED).fit(X_tr, y_tr) for T in (1, 3, 10, 100)}
scores_t = {T: mcc(y_te, m.predict(X_te)) for T, m in bags.items()}
show(
    pl.DataFrame(
        [(T, s, bags[T].oob_mcc(X_tr, y_tr)) for T, s in scores_t.items()],
        schema=["trees B", "test MCC", "OOB MCC"],
        orient="row",
    )
)

In [ ]:
# plot e3_bagging
fg, axs = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.25))
fg.subplots_adjust(left=0.005, right=0.995, top=0.9, bottom=0.11, wspace=0.05)
for ax, (T, m) in zip(axs, bags.items()):
    boundaries.frontier_panel(
        ax, m, X_tr, y_tr, LIM, title=f"{T} tree{'s' if T > 1 else ''}", label=f"MCC {scores_t[T]:.2f}", fontsize=6.3
    )
plotlib.save(fg, "e3_bagging")

In [ ]:
slide("Out-of-Bag MCC Against the Number of Trees")


def oob_curve(seed):
    a, c, b_, d = sy.load("checkerboard", seed)
    with threadpool_limits(1):
        bag = HandBag(200, seed).fit(a, c)
        probs = np.cumsum([t.predict_proba(b_)[:, 1] for t in bag.trees_], axis=0)
        return [
            (T, bag.oob_mcc(a, c, T), mcc(d, probs[T - 1] / T > 0.5)) for T in (2, 3, 5, 10, 20, 30, 50, 100, 150, 200)
        ]


oc = Parallel(n_jobs=N_JOBS)(delayed(oob_curve)(s) for s in range(10))
ocl = pl.DataFrame([r for run in oc for r in run], schema=["T", "oob", "test"], orient="row")
oca = (
    ocl.group_by("T")
    .agg(
        pl.col("oob").mean(),
        pl.col("oob").std().alias("oob sd"),
        pl.col("test").mean(),
        pl.col("test").std().alias("test sd"),
    )
    .sort("T")
)
show(oca)

In [ ]:
# plot e3_oob
fg, ax = plotlib.fig("full", 1.3)
for col, key, label, ls in ((BLUE, "oob", "out-of-bag MCC", "-"), (ORANGE, "test", "test MCC", "--")):
    m, s = oca[key].to_numpy(), oca[key + " sd"].to_numpy()
    ax.plot(oca["T"], m, color=col, ls=ls, marker="o", ms=2.5, label=label)
    ax.fill_between(oca["T"], m - s, m + s, color=col, alpha=0.15, linewidth=0)
ax.set(xscale="log", xlabel="trees $B$ (log scale)", ylabel="MCC")
ax.legend(loc="lower right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e3_oob")

## A random forest decorrelates the trees

Each split chooses among $m$ features drawn at random: the correlation of the trees falls, and each tree gets weaker.

In [ ]:
slide("A Random Forest Decorrelates the Trees: max_features")
D_FEAT = 20


def rho_run(seed, max_features):
    X_, y_ = make_classification(
        1000,
        n_features=D_FEAT,
        n_informative=6,
        n_redundant=4,
        n_clusters_per_class=2,
        flip_y=0.05,
        class_sep=1.0,
        random_state=seed,
    )
    a, b, c, d = data.split(X_, y_, seed)
    with threadpool_limits(1):
        rf = RandomForestClassifier(n_estimators=100, max_features=max_features, random_state=seed).fit(a, c)
    tree_pred = np.array([t.predict(b.astype(np.float32)) for t in rf.estimators_])
    err = (tree_pred != d).astype(float)
    return (
        max_features,
        float(metrics.mean_pairwise_corr(err)),
        float(np.mean([mcc(d, p) for p in tree_pred])),
        mcc(d, rf.predict(b)),
    )


FEATS = [1, 2, 4, 6, 10, 15, 20]
rr = Parallel(n_jobs=N_JOBS)(delayed(rho_run)(s, mf) for s in range(5) for mf in FEATS)
rl = pl.DataFrame(rr, schema=["max_features", "rho", "tree MCC", "forest MCC"], orient="row")
rho_t = rl.group_by("max_features").agg(pl.all().mean()).sort("max_features")
show(rho_t)
print(
    f"d = {D_FEAT} features, 1000 points, 5 seeds; rho = mean pairwise correlation of the 0/1 errors of the 100 trees on the test set"
)
print("max_features = 20 is plain bagging; sqrt(d) = 4 is the default of scikit-learn for classification")

In [ ]:
# plot e3_maxfeat
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.4))
axs[0].plot(rho_t["max_features"], rho_t["rho"], color=BLUE, marker="o", ms=3)
axs[0].set(xlabel="max_features $m$ (of 20)", ylabel="correlation $\\rho$ of two trees' errors", ylim=(0, None))
axs[1].plot(rho_t["max_features"], rho_t["tree MCC"], color=GREY, marker="s", ms=3, ls="--", label="one tree (mean)")
axs[1].plot(rho_t["max_features"], rho_t["forest MCC"], color=ORANGE, marker="o", ms=3, label="forest of 100 trees")
axs[1].set(xlabel="max_features $m$ (of 20)", ylabel="test MCC")
axs[1].legend(loc="lower right", fontsize=6.3, handlelength=1.4)
fg.subplots_adjust(wspace=0.38)
plotlib.save(fg, "e3_maxfeat")

## Bias and variance

The decomposition holds for the squared error, so this one experiment reports MSE; the table also prints the MAE.

In [ ]:
slide("Bias and Variance of a Tree, a Bagged Tree and a Forest")
N_SETS, N_TRAIN, SD, D_BV, CORR = 200, 300, 0.5, 10, 0.5


def draw_x(rng, n):
    """n points of 10 features with pairwise correlation 0.5 (one shared factor): the correlated features of real tables."""
    return np.sqrt(1 - CORR) * rng.normal(size=(n, D_BV)) + np.sqrt(CORR) * rng.normal(size=(n, 1))


def truth(X):
    """2 sin(x1) + 0.6 (tanh(x2) + ... + tanh(x7)): one strong feature, six weak ones, three pure noise columns."""
    return 2 * np.sin(X[:, 0]) + 0.6 * np.tanh(X[:, 1:7]).sum(axis=1)


X_grid = draw_x(np.random.default_rng(99), 500)
f_grid = truth(X_grid)
FORESTS = {"bagged": 1.0, "forest m=5": 5, "forest m=3": 3}


def bv_run(seed):
    rng = np.random.default_rng(seed)
    Xs = draw_x(rng, N_TRAIN)
    ys = truth(Xs) + rng.normal(0, SD, N_TRAIN)
    with threadpool_limits(1):
        out = {"tree": DecisionTreeRegressor(random_state=seed).fit(Xs, ys).predict(X_grid)}
        for name, mf in FORESTS.items():
            out[name] = (
                RandomForestRegressor(n_estimators=50, max_features=mf, random_state=seed).fit(Xs, ys).predict(X_grid)
            )
    return out


runs = Parallel(n_jobs=N_JOBS)(delayed(bv_run)(s) for s in range(N_SETS))
rows = []
for name in ("tree", *FORESTS):
    P = np.array([r[name] for r in runs])  # sets x points
    bias2 = float(np.mean((P.mean(axis=0) - f_grid) ** 2))
    var = float(np.mean(P.var(axis=0)))
    mse = float(np.mean((P - f_grid) ** 2)) + SD**2
    rows.append((name, bias2, var, SD**2, bias2 + var + SD**2, mse, float(np.mean(np.abs(P - f_grid)))))
bv = pl.DataFrame(
    rows, schema=["model", "bias^2", "variance", "noise", "sum", "expected test MSE", "MAE to the truth"], orient="row"
)
show(bv)
print(
    f"{N_SETS} training sets of {N_TRAIN} points, {D_BV} correlated features (rho 0.5, 7 informative), noise sd {SD}; bias^2 and variance are averaged over 500 fixed test points"
)
print(
    "bagged = 50 trees on bootstrap samples with all 10 features per split; forest m = the same with m features drawn at each split"
)
print("the decomposition holds: 'sum' and 'expected test MSE' agree")

In [ ]:
# plot e3_biasvar
fg, ax = plotlib.fig("col", 1.6)
names = bv["model"].to_list()
x = np.arange(len(names))
b2, va, no = bv["bias^2"].to_numpy(), bv["variance"].to_numpy(), bv["noise"].to_numpy()
ax.bar(x, no, color=plotlib.LIGHT, label="noise \u03c3\u00b2", width=0.6)
ax.bar(x, b2, bottom=no, color=BLUE, label="bias\u00b2", width=0.6)
ax.bar(x, va, bottom=no + b2, color=ORANGE, label="variance", width=0.6)
for i in range(len(names)):
    ax.text(x[i], no[i] + b2[i] + va[i] + 0.02, f"{no[i] + b2[i] + va[i]:.2f}", ha="center", va="bottom", fontsize=6.3)
    ax.text(x[i], no[i] + b2[i] + va[i] / 2, f"{va[i]:.2f}", ha="center", va="center", fontsize=5.6)
ax.set(
    xticks=x,
    xticklabels=[n.replace(" ", "\n") for n in names],
    ylabel="expected test MSE",
    ylim=(0, (no + b2 + va).max() * 1.2),
)
ax.grid(axis="x", visible=False)
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, 1.2), fontsize=6.0, handlelength=1.0, columnspacing=0.7)
ax.tick_params(axis="x", labelsize=6.5)
plotlib.save(fg, "e3_biasvar")

In [ ]:
# plot e3_spaghetti: 30 fits of a tree and of a bagged tree on one noisy sine
rng = np.random.default_rng(SEED)
xs = np.linspace(-3, 3, 300)[:, None]
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.35), sharey=True)
fits = {"tree": [], "bagged (25 trees)": []}
for k in range(30):
    x1 = rng.uniform(-3, 3, 60)
    y1 = np.sin(1.5 * x1) + rng.normal(0, 0.3, 60)
    fits["tree"].append(DecisionTreeRegressor(random_state=k).fit(x1[:, None], y1).predict(xs))
    fits["bagged (25 trees)"].append(
        RandomForestRegressor(n_estimators=25, max_features=1.0, random_state=k).fit(x1[:, None], y1).predict(xs)
    )
for ax, (name, P), col in zip(axs, fits.items(), (RED, BLUE)):
    P = np.array(P)
    for p in P:
        ax.plot(xs[:, 0], p, color=col, alpha=0.18, lw=0.7)
    ax.plot(xs[:, 0], np.sin(1.5 * xs[:, 0]), color="k", lw=1.0, ls=":")
    ax.set(xlabel="$x$", title=f"{name}: variance {P.var(axis=0).mean():.3f}", xlim=(-3, 3))
    ax.title.set_fontsize(7)
axs[0].set_ylabel("prediction (30 fits, 60 points each)")
fg.subplots_adjust(wspace=0.08)
plotlib.save(fg, "e3_spaghetti")

In [ ]:
slide("Limits of Bagging: Extrapolation")
x = np.random.default_rng(SEED).uniform(-3, 3, 200)
y = np.sin(2 * x) + np.random.default_rng(SEED + 1).normal(0, 0.1, 200)
xs = np.linspace(-4.5, 4.5, 400)[:, None]
forest = RandomForestRegressor(n_estimators=100, random_state=SEED).fit(x[:, None], y)
out = np.abs(xs[:, 0]) > 3
print(
    f"MAE against the sine: inside the data {np.mean(np.abs(forest.predict(xs[~out]) - np.sin(2 * xs[~out, 0]))):.3f}, outside {np.mean(np.abs(forest.predict(xs[out]) - np.sin(2 * xs[out, 0]))):.3f}"
)

In [ ]:
# plot e3_extrap
fg, ax = plotlib.fig("full", 1.2)
ax.axvspan(-3, 3, color="#EEF1F4", lw=0, zorder=0)
ax.scatter(x, y, s=2.5, color=ORANGE, linewidths=0, zorder=2)
ax.plot(xs[:, 0], np.sin(2 * xs[:, 0]), color=boundaries.LINE, lw=0.6, ls=":", label="true function")
ax.plot(xs[:, 0], forest.predict(xs), color=BLUE, lw=1.0, label="random forest")
ax.set(xlabel="$x$ (shaded: range of the training data)", ylabel="$y$", xlim=(-4.5, 4.5))
ax.legend(ncols=2, loc="upper center", bbox_to_anchor=(0.5, 1.2), fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e3_extrap")

In [ ]:
slide("Feature Importance: Impurity Against Permutation")
rng = np.random.default_rng(SEED)
Xm, ym = make_classification(
    1000, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, flip_y=0.05, random_state=SEED
)
copy = Xm[:, 0] + rng.normal(0, 0.15, len(ym))
extra = np.c_[copy, rng.normal(size=len(ym)), rng.integers(0, 10, len(ym)), rng.integers(0, 2, len(ym))]
Xi = np.c_[Xm, extra]
FEAT = ["x1", "x2", "x1 copy", "random continuous", "random 0-9", "random 0/1"]
a, b, c, d = data.split(Xi, ym, SEED)
with threadpool_limits(1):
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(a, c)
    perm = permutation_importance(rf, b, d, scoring="matthews_corrcoef", n_repeats=30, random_state=SEED)
mdi = rf.feature_importances_
imp = pl.DataFrame(
    {
        "feature": FEAT,
        "impurity (train)": mdi,
        "permutation (test)": perm.importances_mean,
        "perm sd": perm.importances_std,
    }
)
show(imp)
print(
    f"forest test MCC {mcc(d, rf.predict(b)):.3f}; the three noise columns get {mdi[3:].sum():.3f} of the impurity importance and {perm.importances_mean[3:].sum():.3f} (MCC drop) of the permutation importance"
)
print(
    f"shares: the three random columns hold {mdi[3:].sum():.1%} of the impurity importance and {perm.importances_mean[3:].sum() / perm.importances_mean.sum():.1%} of the permutation importance"
)
print(
    f"x1 and its noisy copy share the signal: permuting x1 alone costs {perm.importances_mean[0]:.3f}; the copy {perm.importances_mean[2]:.3f}"
)

In [ ]:
# plot e3_importance
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.5), sharey=True)
y_pos = np.arange(len(FEAT))[::-1]
axs[0].barh(y_pos, mdi, color=BLUE, height=0.6)
axs[0].set(xlabel="impurity importance")
axs[1].barh(y_pos, perm.importances_mean, xerr=perm.importances_std, color=ORANGE, height=0.6, error_kw={"lw": 0.6})
axs[1].set(xlabel="MCC drop when permuted")
axs[0].set_yticks(y_pos, FEAT)
for ax in axs:
    ax.grid(axis="y", visible=False)
fg.subplots_adjust(wspace=0.12)
plotlib.save(fg, "e3_importance")

## Bagging helps unstable learners only

In [ ]:
slide("Bagging Helps Unstable Learners Only")


def stability_run(seed):
    a, c, b, d = sy.load("spiral", seed)
    g = ga.classifiers(seed)
    ms = {
        "deep tree": DecisionTreeClassifier(random_state=seed),
        "bagged deep tree": BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, random_state=seed),
        "15-NN": g["15-NN"],
        "bagged 15-NN": BaggingClassifier(g["15-NN"], n_estimators=50, random_state=seed),
        "1-NN": g["1-NN"],
        "bagged 1-NN": BaggingClassifier(g["1-NN"], n_estimators=50, random_state=seed),
        "logistic": g["logistic"],
        "bagged logistic": BaggingClassifier(g["logistic"], n_estimators=50, random_state=seed),
    }
    with threadpool_limits(1):
        return [(n, seed, mcc(d, m.fit(a, c).predict(b))) for n, m in ms.items()]


sr = Parallel(n_jobs=N_JOBS)(delayed(stability_run)(s) for s in range(10))
sl = pl.DataFrame([r for run in sr for r in run], schema=["model", "seed", "MCC"], orient="row")
order = ["deep tree", "bagged deep tree", "15-NN", "bagged 15-NN", "1-NN", "bagged 1-NN", "logistic", "bagged logistic"]
sa = (
    sl.group_by("model")
    .agg(pl.col("MCC").mean(), pl.col("MCC").std().alias("sd"))
    .join(pl.DataFrame({"model": order, "i": range(8)}), on="model")
    .sort("i")
    .drop("i")
)
show(sa)
for single_name in ("deep tree", "15-NN", "1-NN", "logistic"):
    delta = (
        sl.filter(pl.col("model") == f"bagged {single_name}")["MCC"].to_numpy()
        - sl.filter(pl.col("model") == single_name)["MCC"].to_numpy()
    )
    print(
        f"bagging {single_name}: mean gain {delta.mean():+.3f} (sd over seeds {delta.std(ddof=1):.3f}, {int(np.sum(delta > 0))} of 10 seeds better)"
    )

## Bagging against the shallow models and the votes

The same figures as Demo 07b, with bagged trees and a random forest added.

In [ ]:
slide("Bagging: the Board")
F = ["shallow", "voting", "bagging"]
clf = ga.evaluate(CLASS_SETS, names_of("c", F))
show(ga.wide(clf, names_of("c", F), CLASS_SETS))

In [ ]:
reg = ga.evaluate(REG_SETS, names_of("r", F))
show(ga.wide(reg, names_of("r", F), REG_SETS))

In [ ]:
earlier_c, earlier_r = ga.SHALLOW_C + ga.VOTING_C, ga.SHALLOW_R + ga.VOTING_R
show(ga.gain(clf, CLASS_SETS, ga.BAGGING_C, earlier_c, "c"))
show(ga.gain(reg, REG_SETS, ga.BAGGING_R, earlier_r, "r"))

In [ ]:
rows = {"shallow": ga.SHALLOW_C, "voting": ga.VOTING_C, "trees": ga.BAGGING_C}
plotlib.save(family_figure("spiral", rows, {"voting": MEMBERS["c"]}), "e3_gallery_spiral")

In [ ]:
rows = {"shallow": ga.SHALLOW_C, "voting": ga.VOTING_C, "trees": ga.BAGGING_C}
plotlib.save(family_figure("checkerboard", rows, {"voting": MEMBERS["c"]}), "e3_gallery_checker")

In [ ]:
rows = {"shallow": ga.SHALLOW_R, "voting": ga.VOTING_R, "trees": ga.BAGGING_R}
plotlib.save(family_figure("sawtooth", rows, {"voting": MEMBERS["r"]}), "e3_gallery_saw")

In [ ]:
rows = {"shallow": ga.SHALLOW_R, "voting": ga.VOTING_R, "trees": ga.BAGGING_R}
plotlib.save(family_figure("chirp", rows, {"voting": MEMBERS["r"]}), "e3_gallery_chirp")

In [ ]:
plotlib.save(board(clf, "c", CLASS_SETS), "e3_board_c")

In [ ]:
plotlib.save(board(reg, "r", REG_SETS), "e3_board_r")